In [10]:
import pandas as pd
import numpy as np

print("Python is working!")

Python is working!


In [11]:
ratings_sample = pd.read_csv(
    "data/ratings.csv",
    nrows=10
)

ratings_sample


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858
5,1,34,2.0,943228491
6,1,36,1.0,944249008
7,1,80,5.0,944248943
8,1,110,3.0,943231119
9,1,111,5.0,944249008


In [12]:
movies = pd.read_csv("data/movies.csv")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [13]:
ratings_iter = pd.read_csv(
    "data/ratings.csv",
    chunksize=500000
)

for chunk in ratings_iter:
    print(chunk.shape)
    break

(500000, 4)


In [14]:
# Check basic statistics of the full ratings dataset

total_ratings = 0
unique_users = set()
unique_movies = set()
missing_values = 0
duplicate_ratings = 0

for chunk in pd.read_csv("data/ratings.csv", chunksize=500000):
    
    total_ratings += len(chunk)
    
    unique_users.update(chunk["userId"].unique())
    unique_movies.update(chunk["movieId"].unique())
    
    missing_values += chunk.isnull().sum().sum()
    duplicate_ratings += chunk.duplicated().sum()

print("Total ratings:", total_ratings)
print("Unique users:", len(unique_users))
print("Unique movies:", len(unique_movies))
print("Missing values:", missing_values)
print("Duplicate rows:", duplicate_ratings)

Total ratings: 32000204
Unique users: 200948
Unique movies: 84432
Missing values: 0
Duplicate rows: 0


In [15]:
# Rating distribution

rating_counts = {}

for chunk in pd.read_csv("data/ratings.csv", chunksize=500000):
    
    counts = chunk["rating"].value_counts()
    
    for rating, count in counts.items():
        rating_counts[rating] = rating_counts.get(rating, 0) + count

rating_distribution = pd.Series(rating_counts).sort_index()

print(rating_distribution)

0.5     525132
1.0     946675
1.5     531063
2.0    2028622
2.5    1685386
3.0    6054990
3.5    4290105
4.0    8367654
4.5    2974000
5.0    4596577
dtype: int64


In [16]:
print("Number of movies:", len(movies))
print("Missing values:")
print(movies.isnull().sum())

print("\nMovie genres:")
print(movies["genres"].head(10))

Number of movies: 87585
Missing values:
movieId    0
title      0
genres     0
dtype: int64

Movie genres:
0    Adventure|Animation|Children|Comedy|Fantasy
1                     Adventure|Children|Fantasy
2                                 Comedy|Romance
3                           Comedy|Drama|Romance
4                                         Comedy
5                          Action|Crime|Thriller
6                                 Comedy|Romance
7                             Adventure|Children
8                                         Action
9                      Action|Adventure|Thriller
Name: genres, dtype: object


In [17]:
# Count ratings per user

user_rating_counts = {}

for chunk in pd.read_csv("data/ratings.csv", chunksize=500000):
    
    counts = chunk["userId"].value_counts()
    
    for user_id, count in counts.items():
        user_rating_counts[user_id] = user_rating_counts.get(user_id, 0) + count

user_rating_counts = pd.Series(user_rating_counts)

print(user_rating_counts.describe())

count    200948.000000
mean        159.246193
std         282.025462
min          20.000000
25%          36.000000
50%          73.000000
75%         167.000000
max       33332.000000
dtype: float64


In [18]:
# Count ratings per movie

movie_rating_counts = {}

for chunk in pd.read_csv("data/ratings.csv", chunksize=500000):
    
    counts = chunk["movieId"].value_counts()
    
    for movie_id, count in counts.items():
        movie_rating_counts[movie_id] = movie_rating_counts.get(movie_id, 0) + count

movie_rating_counts = pd.Series(movie_rating_counts)

print(movie_rating_counts.describe())

count     84432.000000
mean        379.005638
std        2592.439791
min           1.000000
25%           2.000000
50%           5.000000
75%          25.000000
max      102929.000000
dtype: float64


In [19]:
print("Movies with at least 10 ratings:",
      (movie_rating_counts >= 10).sum())

print("Movies with at least 50 ratings:",
      (movie_rating_counts >= 50).sum())

print("Movies with at least 100 ratings:",
      (movie_rating_counts >= 100).sum())

print("Movies with at least 500 ratings:",
      (movie_rating_counts >= 500).sum())

print("Movies with at least 1000 ratings:",
      (movie_rating_counts >= 1000).sum())

Movies with at least 10 ratings: 31961
Movies with at least 50 ratings: 16034
Movies with at least 100 ratings: 12191
Movies with at least 500 ratings: 6226
Movies with at least 1000 ratings: 4397


In [20]:
MIN_USER_RATINGS = 50
MIN_MOVIE_RATINGS = 50

valid_users = set(
    user_rating_counts[
        user_rating_counts >= MIN_USER_RATINGS
    ].index
)

valid_movies = set(
    movie_rating_counts[
        movie_rating_counts >= MIN_MOVIE_RATINGS
    ].index
)

print("Users with at least 50 ratings:", len(valid_users))
print("Movies with at least 50 ratings:", len(valid_movies))

Users with at least 50 ratings: 128344
Movies with at least 50 ratings: 16034


In [21]:
filtered_file = "data/ratings_filtered.csv"

first_chunk = True
filtered_count = 0

for chunk in pd.read_csv(
    "data/ratings.csv",
    chunksize=500000
):
    
    filtered_chunk = chunk[
        chunk["userId"].isin(valid_users) &
        chunk["movieId"].isin(valid_movies)
    ]
    
    filtered_count += len(filtered_chunk)
    
    filtered_chunk.to_csv(
        filtered_file,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )
    
    first_chunk = False

print("Filtered ratings:", filtered_count)

Filtered ratings: 29210880


In [22]:
filtered_ratings = pd.read_csv(
    "data/ratings_filtered.csv"
)

print("Filtered ratings:", len(filtered_ratings))
print("Filtered users:", filtered_ratings["userId"].nunique())
print("Filtered movies:", filtered_ratings["movieId"].nunique())
print("Sparsity:", 1 - (
    len(filtered_ratings) /
    (
        filtered_ratings["userId"].nunique()
        *
        filtered_ratings["movieId"].nunique()
    )
))

Filtered ratings: 29210880
Filtered users: 128344
Filtered movies: 16034
Sparsity: 0.985805268211956


In [23]:
# Check the actual number of ratings per user after filtering

final_user_counts = filtered_ratings["userId"].value_counts()

print("Minimum user ratings after filtering:", final_user_counts.min())
print("Median user ratings:", final_user_counts.median())
print("Users with fewer than 50 ratings:", (final_user_counts < 50).sum())

Minimum user ratings after filtering: 10
Median user ratings: 128.0
Users with fewer than 50 ratings: 260


In [24]:
# Check the actual number of ratings per movie after filtering

final_movie_counts = filtered_ratings["movieId"].value_counts()

print("Minimum movie ratings after filtering:", final_movie_counts.min())
print("Median movie ratings:", final_movie_counts.median())
print("Movies with fewer than 50 ratings:", (final_movie_counts < 50).sum())

Minimum movie ratings after filtering: 37
Median movie ratings: 275.0
Movies with fewer than 50 ratings: 142


In [25]:
# Iterative filtering: keep users and movies with at least 50 ratings

MIN_RATINGS = 50

iteration = 0

while True:
    iteration += 1
    
    # Count ratings for each user
    user_counts = filtered_ratings["userId"].value_counts()
    
    # Count ratings for each movie
    movie_counts = filtered_ratings["movieId"].value_counts()
    
    # Keep users and movies meeting the threshold
    valid_users_iter = user_counts[user_counts >= MIN_RATINGS].index
    valid_movies_iter = movie_counts[movie_counts >= MIN_RATINGS].index
    
    # Filter the dataset
    new_filtered_ratings = filtered_ratings[
        filtered_ratings["userId"].isin(valid_users_iter) &
        filtered_ratings["movieId"].isin(valid_movies_iter)
    ].copy()
    
    print(
        f"Iteration {iteration}: "
        f"{len(new_filtered_ratings):,} ratings, "
        f"{new_filtered_ratings['userId'].nunique():,} users, "
        f"{new_filtered_ratings['movieId'].nunique():,} movies"
    )
    
    # Stop when no more rows are removed
    if len(new_filtered_ratings) == len(filtered_ratings):
        break
    
    filtered_ratings = new_filtered_ratings

Iteration 1: 29,192,104 ratings, 128,084 users, 15,892 movies
Iteration 2: 29,191,222 ratings, 128,075 users, 15,883 movies
Iteration 3: 29,191,222 ratings, 128,075 users, 15,883 movies


In [26]:
# Final check after iterative filtering

final_user_counts = filtered_ratings["userId"].value_counts()
final_movie_counts = filtered_ratings["movieId"].value_counts()

print("===== FINAL DATASET =====")

print("Ratings:", len(filtered_ratings))
print("Users:", filtered_ratings["userId"].nunique())
print("Movies:", filtered_ratings["movieId"].nunique())

print("\n===== USER CHECK =====")
print("Minimum user ratings:", final_user_counts.min())
print("Users below 50:", (final_user_counts < 50).sum())

print("\n===== MOVIE CHECK =====")
print("Minimum movie ratings:", final_movie_counts.min())
print("Movies below 50:", (final_movie_counts < 50).sum())

===== FINAL DATASET =====
Ratings: 29191222
Users: 128075
Movies: 15883

===== USER CHECK =====
Minimum user ratings: 50
Users below 50: 0

===== MOVIE CHECK =====
Minimum movie ratings: 50
Movies below 50: 0


In [27]:
# Save the final preprocessed dataset

final_file = "data/ratings_final.csv"

filtered_ratings.to_csv(
    final_file,
    index=False
)

print("Final dataset saved successfully.")
print("File:", final_file)

Final dataset saved successfully.
File: data/ratings_final.csv


In [28]:
# Rating distribution of the final dataset

final_rating_distribution = (
    filtered_ratings["rating"]
    .value_counts()
    .sort_index()
)

print(final_rating_distribution)

rating
0.5     464502
1.0     856864
1.5     489186
2.0    1870342
2.5    1572134
3.0    5510885
3.5    4025786
4.0    7662865
4.5    2738612
5.0    4000046
Name: count, dtype: int64


In [29]:
# Basic statistics of final ratings

print(filtered_ratings["rating"].describe())

count    2.919122e+07
mean     3.531619e+00
std      1.051511e+00
min      5.000000e-01
25%      3.000000e+00
50%      3.500000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64


In [30]:
# Train-Test Split

from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    filtered_ratings,
    test_size=0.20,
    random_state=42
)

print("Training ratings:", len(train_data))
print("Testing ratings:", len(test_data))

Training ratings: 23352977
Testing ratings: 5838245


In [31]:
print("Training users:", train_data["userId"].nunique())
print("Training movies:", train_data["movieId"].nunique())

print("Testing users:", test_data["userId"].nunique())
print("Testing movies:", test_data["movieId"].nunique())

Training users: 128075
Training movies: 15883
Testing users: 128075
Testing movies: 15883


In [32]:
from scipy.sparse import csr_matrix
import numpy as np

# Create mappings from original IDs to matrix indices

user_ids = train_data["userId"].unique()
movie_ids = train_data["movieId"].unique()

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

movie_to_index = {
    movie_id: index
    for index, movie_id in enumerate(movie_ids)
}

# Convert IDs into matrix indices

row_indices = train_data["userId"].map(user_to_index).values
col_indices = train_data["movieId"].map(movie_to_index).values

ratings_values = train_data["rating"].values.astype(np.float32)

# Create sparse User-Movie matrix

user_movie_matrix = csr_matrix(
    (
        ratings_values,
        (row_indices, col_indices)
    ),
    shape=(len(user_ids), len(movie_ids))
)

print("Matrix shape:", user_movie_matrix.shape)
print("Number of stored ratings:", user_movie_matrix.nnz)

Matrix shape: (128075, 15883)
Number of stored ratings: 23352977


In [33]:
item_user_matrix = user_movie_matrix.T.tocsr()

print("Item-User matrix shape:", item_user_matrix.shape)
print("Stored ratings:", item_user_matrix.nnz)

Item-User matrix shape: (15883, 128075)
Stored ratings: 23352977


In [34]:
from sklearn.neighbors import NearestNeighbors

# Create Item-Based Collaborative Filtering model

item_cf_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11,
    n_jobs=-1
)

item_cf_model.fit(item_user_matrix)

print("Item-Based Collaborative Filtering model trained successfully.")

Item-Based Collaborative Filtering model trained successfully.


In [35]:
# Find a movie by title

movies[movies["title"].str.contains(
    "Toy Story",
    case=False,
    na=False
)].head(10)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3021,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
14815,78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX
20505,106022,Toy Story of Terror (2013),Animation|Children|Comedy
22646,115875,Toy Story Toons: Hawaiian Vacation (2011),Adventure|Animation|Children|Comedy|Fantasy
22647,115879,Toy Story Toons: Small Fry (2011),Adventure|Animation|Children|Comedy|Fantasy
24088,120468,Toy Story Toons: Partysaurus Rex (2012),Animation|Children|Comedy
24090,120474,Toy Story That Time Forgot (2014),Animation|Children
60708,201588,Toy Story 4 (2019),Adventure|Animation|Children|Comedy


In [36]:
# Get the matrix index of Toy Story

toy_story_id = 1

toy_story_index = movie_to_index[toy_story_id]

distances, indices = item_cf_model.kneighbors(
    item_user_matrix[toy_story_index],
    n_neighbors=11
)

print("Distances:")
print(distances)

print("\nIndices:")
print(indices)

Distances:
[[2.5117397e-04 4.9416858e-01 5.0683844e-01 5.1095480e-01 5.1314312e-01
  5.1585793e-01 5.1755005e-01 5.2535552e-01 5.2910978e-01 5.3289664e-01
  5.4000163e-01]]

Indices:
[[1217   89  149 1185  607 1575  423  701   16  228  473]]


In [37]:
# Create reverse mapping from matrix index to movie ID

index_to_movie = {
    index: movie_id
    for movie_id, index in movie_to_index.items()
}

# Build similar movie results

similar_movies = []

for distance, index in zip(distances[0][1:], indices[0][1:]):
    
    movie_id = index_to_movie[index]
    
    similarity = 1 - distance
    
    movie_title = movies.loc[
        movies["movieId"] == movie_id,
        "title"
    ].iloc[0]
    
    similar_movies.append({
        "movieId": movie_id,
        "title": movie_title,
        "similarity": similarity
    })

similar_movies_df = pd.DataFrame(similar_movies)

similar_movies_df

,movieId,title,similarity
0,3114,Toy Story 2 (1999),0.505831
1,356,Forrest Gump (1994),0.493162
2,260,Star Wars: Episode IV - A New Hope (1977),0.489045
3,480,Jurassic Park (1993),0.486857
4,1270,Back to the Future (1985),0.484142
5,364,"Lion King, The (1994)",0.482450
6,1210,Star Wars: Episode VI - Return of the Jedi (1983),0.474644
7,588,Aladdin (1992),0.470890
8,1196,Star Wars: Episode V - The Empire Strikes Back...,0.467103
9,4306,Shrek (2001),0.459998


In [38]:
def recommend_movies(user_id, n=10, top_k_similar=10):
    
    # Check whether user exists
    if user_id not in user_to_index:
        return pd.DataFrame(
            columns=["movieId", "title", "score"]
        )
    
    # Get user's training history
    user_history = train_data[
        train_data["userId"] == user_id
    ]
    
    if user_history.empty:
        return pd.DataFrame(
            columns=["movieId", "title", "score"]
        )
    
    # Movies already rated by the user
    rated_movies = set(user_history["movieId"])
    
    # Keep only movies that exist in the training matrix
    history = user_history[
        user_history["movieId"].isin(movie_to_index)
    ].copy()
    
    # Get matrix indices for the user's rated movies
    history_indices = [
        movie_to_index[movie_id]
        for movie_id in history["movieId"]
    ]
    
    # Get item vectors for all movies rated by the user
    history_matrix = item_user_matrix[history_indices]
    
    # Find similar movies for all history movies at once
    distances, indices = item_cf_model.kneighbors(
        history_matrix,
        n_neighbors=top_k_similar + 1
    )
    
    # Store recommendation scores
    recommendation_scores = {}
    
    # Calculate scores
    for row_idx, (_, history_row) in enumerate(history.iterrows()):
        
        user_rating = history_row["rating"]
        
        # Skip the first result because it is the movie itself
        for distance, similar_index in zip(
            distances[row_idx][1:],
            indices[row_idx][1:]
        ):
            
            similar_movie_id = index_to_movie[similar_index]
            
            # Do not recommend movies already rated by the user
            if similar_movie_id in rated_movies:
                continue
            
            # Convert cosine distance to cosine similarity
            similarity = 1 - distance
            
            # Weighted recommendation score
            score_contribution = similarity * user_rating
            
            recommendation_scores[similar_movie_id] = (
                recommendation_scores.get(similar_movie_id, 0)
                + score_contribution
            )
    
    # Convert scores into a DataFrame
    recommendations = []

    for movie_id, score in recommendation_scores.items():
        
        recommendations.append({
            "movieId": movie_id,
            "score": score
        })
    
    recommendations_df = pd.DataFrame(recommendations)
    
    # No recommendations available
    if recommendations_df.empty:
        return recommendations_df
    
    # Sort by recommendation score
    recommendations_df = (
        recommendations_df
        .sort_values(
            "score",
            ascending=False
        )
        .head(n)
    )
    
    # Add movie titles
    recommendations_df = recommendations_df.merge(
        movies[["movieId", "title"]],
        on="movieId",
        how="left"
    )
    
    return recommendations_df[
        ["movieId", "title", "score"]
    ]

In [39]:
recommendations_user2 = recommend_movies(2, n=10)

recommendations_user2

,movieId,title,score
0,377,Speed (1994),23.819615
1,587,Ghost (1990),22.531308
2,339,While You Were Sleeping (1995),19.614968
3,480,Jurassic Park (1993),14.528489
4,588,Aladdin (1992),13.929888
5,380,True Lies (1994),13.296223
6,364,"Lion King, The (1994)",13.132167
7,590,Dances with Wolves (1990),12.259531
8,292,Outbreak (1995),11.763840
9,367,"Mask, The (1994)",11.658019


In [40]:
rated_by_user2 = set(
    train_data[
        train_data["userId"] == 2
    ]["movieId"]
)

already_rated = (
    set(recommendations_user2["movieId"])
    & rated_by_user2
)

print("Already rated movies in recommendations:")
print(already_rated)

Already rated movies in recommendations:
set()


In [41]:
recommendations_user1 = recommend_movies(1, n=10)

recommendations_user1

,movieId,title,score
0,296,Pulp Fiction (1994),23.911409
1,1198,Raiders of the Lost Ark (Indiana Jones and the...,18.056648
2,1240,"Terminator, The (1984)",17.715012
3,1214,Alien (1979),16.320241
4,750,Dr. Strangelove or: How I Learned to Stop Worr...,16.190784
5,2571,"Matrix, The (1999)",16.010250
6,1193,One Flew Over the Cuckoo's Nest (1975),15.879601
7,1252,Chinatown (1974),15.693319
8,1089,Reservoir Dogs (1992),15.012072
9,924,2001: A Space Odyssey (1968),14.399661


In [42]:
recommend_movies(1, n=10)

,movieId,title,score
0,296,Pulp Fiction (1994),23.911409
1,1198,Raiders of the Lost Ark (Indiana Jones and the...,18.056648
2,1240,"Terminator, The (1984)",17.715012
3,1214,Alien (1979),16.320241
4,750,Dr. Strangelove or: How I Learned to Stop Worr...,16.190784
5,2571,"Matrix, The (1999)",16.010250
6,1193,One Flew Over the Cuckoo's Nest (1975),15.879601
7,1252,Chinatown (1974),15.693319
8,1089,Reservoir Dogs (1992),15.012072
9,924,2001: A Space Odyssey (1968),14.399661


In [43]:
# Check User 1's rating history

user_1_history = train_data[
    train_data["userId"] == 1
].merge(
    movies[["movieId", "title"]],
    on="movieId",
    how="left"
)

print("Number of movies rated by User 1:",
      len(user_1_history))

user_1_history[
    ["movieId", "title", "rating"]
].sort_values(
    "rating",
    ascending=False
).head(20)

Number of movies rated by User 1: 120


,movieId,title,rating
60,1200,Aliens (1986),5.0
36,1203,12 Angry Men (1957),5.0
73,1748,Dark City (1998),5.0
70,2336,Elizabeth (1998),5.0
69,1939,"Best Years of Our Lives, The (1946)",5.0
67,1090,Platoon (1986),5.0
66,2985,RoboCop (1987),5.0
62,2973,Crimes and Misdemeanors (1989),5.0
59,1208,Apocalypse Now (1979),5.0
57,166,"Doom Generation, The (1995)",5.0


In [44]:
print(
    user_1_history["rating"]
    .value_counts()
    .sort_index()
)

rating
1.0    24
2.0    11
3.0    15
4.0    22
5.0    48
Name: count, dtype: int64


In [45]:
recommend_movies(2, n=10)

,movieId,title,score
0,377,Speed (1994),23.819615
1,587,Ghost (1990),22.531308
2,339,While You Were Sleeping (1995),19.614968
3,480,Jurassic Park (1993),14.528489
4,588,Aladdin (1992),13.929888
5,380,True Lies (1994),13.296223
6,364,"Lion King, The (1994)",13.132167
7,590,Dances with Wolves (1990),12.259531
8,292,Outbreak (1995),11.763840
9,367,"Mask, The (1994)",11.658019


In [46]:
recommend_movies(10, n=10)

,movieId,title,score
0,59315,Iron Man (2008),37.753672
1,1198,Raiders of the Lost Ark (Indiana Jones and the...,34.833134
2,380,True Lies (1994),29.676350
3,356,Forrest Gump (1994),29.625936
4,102125,Iron Man 3 (2013),25.780720
5,153,Batman Forever (1995),21.795673
6,122892,Avengers: Age of Ultron (2015),20.912127
7,1036,Die Hard (1988),20.840741
8,109487,Interstellar (2014),20.784217
9,44191,V for Vendetta (2006),19.526771


In [47]:
# Fast lookup for each user's training history

user_history_indices = train_data.groupby(
    "userId",
    sort=False
).groups

print("Users indexed:", len(user_history_indices))

Users indexed: 128075


In [48]:
# Define relevant items in the testing set
# A rating of 4.0 or above is considered a positive preference

relevant_test = test_data[
    test_data["rating"] >= 4.0
]

print("Relevant test ratings:", len(relevant_test))
print("Users with relevant test ratings:",
      relevant_test["userId"].nunique())

Relevant test ratings: 2881234
Users with relevant test ratings: 127710


In [49]:
# Select users who have at least one relevant item in the test set

evaluation_users = relevant_test["userId"].unique()

np.random.seed(42)

if len(evaluation_users) > 1000:
    evaluation_users = np.random.choice(
        evaluation_users,
        size=1000,
        replace=False
    )

print("Evaluation users:", len(evaluation_users))

Evaluation users: 1000


In [50]:
def recommend_movies_fast(user_id, n=10, top_k_similar=10):
    
    # Check whether user exists
    if user_id not in user_history_indices:
        return []
    
    # Get user's training rows directly
    row_indices = user_history_indices[user_id]
    
    user_history = train_data.loc[
        row_indices,
        ["movieId", "rating"]
    ]
    
    # Movies already rated by the user
    rated_movies = set(
        user_history["movieId"]
    )
    
    # Keep only movies available in the item matrix
    history = user_history[
        user_history["movieId"].isin(movie_to_index)
    ]
    
    if history.empty:
        return []
    
    # Convert movie IDs to matrix indices
    history_indices = np.array([
        movie_to_index[movie_id]
        for movie_id in history["movieId"]
    ])
    
    # Item vectors for user's history
    history_matrix = item_user_matrix[
        history_indices
    ]
    
    # Find similar movies in batch
    distances, indices = item_cf_model.kneighbors(
        history_matrix,
        n_neighbors=top_k_similar + 1
    )
    
    recommendation_scores = {}
    
    # Calculate recommendation score
    for row_idx in range(len(history)):
        
        user_rating = history.iloc[
            row_idx
        ]["rating"]
        
        for distance, similar_index in zip(
            distances[row_idx][1:],
            indices[row_idx][1:]
        ):
            
            similar_movie_id = index_to_movie[
                similar_index
            ]
            
            # Do not recommend movies already rated
            if similar_movie_id in rated_movies:
                continue
            
            similarity = 1 - distance
            
            score = similarity * user_rating
            
            recommendation_scores[
                similar_movie_id
            ] = (
                recommendation_scores.get(
                    similar_movie_id,
                    0
                ) + score
            )
    
    # Sort by recommendation score
    ranked_movies = sorted(
        recommendation_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )
    
    # Return Top-N movie IDs
    return [
        movie_id
        for movie_id, score in ranked_movies[:n]
    ]

In [51]:
def evaluate_recommender(
    evaluation_users,
    k=10
):
    
    precisions = []
    recalls = []
    f1_scores = []
    
    evaluated_users = 0
    
    for user_id in evaluation_users:
        
        # Actual relevant movies in the testing set
        actual_items = set(
            relevant_test[
                relevant_test["userId"] == user_id
            ]["movieId"]
        )
        
        if len(actual_items) == 0:
            continue
        
        # Generate recommendations
        recommended_items = set(
            recommend_movies_precomputed(
                user_id,
                n=k,
                top_k_similar=10
            )
        )
        
        if len(recommended_items) == 0:
            continue
        
        # Calculate hits
        hits = (
            recommended_items
            & actual_items
        )
        
        # Precision
        precision = (
            len(hits)
            / len(recommended_items)
        )
        
        # Recall
        recall = (
            len(hits)
            / len(actual_items)
        )
        
        # F1
        if precision + recall > 0:
            f1 = (
                2 * precision * recall
                / (precision + recall)
            )
        else:
            f1 = 0
        
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        
        evaluated_users += 1
    
    return {
        "Users Evaluated": evaluated_users,
        "Precision@10": np.mean(precisions),
        "Recall@10": np.mean(recalls),
        "F1@10": np.mean(f1_scores)
    }

In [52]:
# Small evaluation sample for performance testing

np.random.seed(42)

evaluation_users_small = np.random.choice(
    relevant_test["userId"].unique(),
    size=50,
    replace=False
)

print("Evaluation users:", len(evaluation_users_small))

Evaluation users: 50


In [54]:
evaluation_results_small = evaluate_recommender(
    evaluation_users_small,
    k=10
)

evaluation_results_small

NameError: name 'recommend_movies_precomputed' is not defined

In [ ]:
# Precompute Top-K similar movies for every movie
# This avoids repeatedly calling kneighbors() during evaluation

TOP_K_SIMILAR = 10

print("Precomputing item-item similarities...")

all_item_distances, all_item_indices = item_cf_model.kneighbors(
    item_user_matrix,
    n_neighbors=TOP_K_SIMILAR + 1
)

print("Precomputation completed.")
print("Shape of distances:", all_item_distances.shape)
print("Shape of indices:", all_item_indices.shape)

In [ ]:
def recommend_movies_precomputed(
    user_id,
    n=10,
    top_k_similar=10
):
    
    # Check whether user exists
    if user_id not in user_history_indices:
        return []
    
    # Get user's training history
    row_indices = user_history_indices[user_id]
    
    user_history = train_data.loc[
        row_indices,
        ["movieId", "rating"]
    ]
    
    # Movies already rated by the user
    rated_movies = set(
        user_history["movieId"]
    )
    
    recommendation_scores = {}
    
    # Process each movie rated by the user
    for movie_id, user_rating in zip(
        user_history["movieId"],
        user_history["rating"]
    ):
        
        # Skip movies not present in matrix
        if movie_id not in movie_to_index:
            continue
        
        movie_index = movie_to_index[movie_id]
        
        # Get precomputed similar movies
        distances = all_item_distances[
            movie_index
        ][1:top_k_similar + 1]
        
        indices = all_item_indices[
            movie_index
        ][1:top_k_similar + 1]
        
        for distance, similar_index in zip(
            distances,
            indices
        ):
            
            similar_movie_id = index_to_movie[
                similar_index
            ]
            
            # Don't recommend movies already rated
            if similar_movie_id in rated_movies:
                continue
            
            # Cosine similarity
            similarity = 1 - distance
            
            # Recommendation score
            score = similarity * user_rating
            
            recommendation_scores[
                similar_movie_id
            ] = (
                recommendation_scores.get(
                    similar_movie_id,
                    0
                ) + score
            )
    
    # Sort by recommendation score
    ranked_movies = sorted(
        recommendation_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )
    
    return [
        movie_id
        for movie_id, score
        in ranked_movies[:n]
    ]

In [ ]:
evaluation_results_small = evaluate_recommender(
    evaluation_users_small,
    k=10
)

evaluation_results_small

In [ ]:
# Final evaluation sample
# Use 1,000 users for a more reliable evaluation

np.random.seed(42)

evaluation_users_final = np.random.choice(
    relevant_test["userId"].unique(),
    size=1000,
    replace=False
)

print(
    "Final evaluation users:",
    len(evaluation_users_final)
)

In [ ]:
# Run final evaluation

evaluation_results_final = evaluate_recommender(
    evaluation_users_final,
    k=10
)

evaluation_results_final

In [ ]:
# Popularity-Based Baseline
# Calculate movie popularity using TRAINING data only

movie_popularity = (
    train_data["movieId"]
    .value_counts()
    .sort_values(ascending=False)
)

print("Number of movies:", len(movie_popularity))
print("\nTop 10 most popular movies:")
print(movie_popularity.head(10))

In [ ]:
def recommend_popular_movies(user_id, n=10):
    
    # Get movies already rated by the user
    if user_id in user_history_indices:
        row_indices = user_history_indices[user_id]
        
        rated_movies = set(
            train_data.loc[
                row_indices,
                "movieId"
            ]
        )
    else:
        rated_movies = set()
    
    recommendations = []
    
    # Go through movies from most popular to least popular
    for movie_id in movie_popularity.index:
        
        # Don't recommend movies already rated
        if movie_id in rated_movies:
            continue
        
        recommendations.append(movie_id)
        
        if len(recommendations) >= n:
            break
    
    return recommendations

In [ ]:
def evaluate_popularity_baseline(
    evaluation_users,
    k=10
):
    
    precisions = []
    recalls = []
    f1_scores = []
    
    evaluated_users = 0
    
    for user_id in evaluation_users:
        
        # Actual relevant movies in testing set
        actual_items = set(
            relevant_test[
                relevant_test["userId"] == user_id
            ]["movieId"]
        )
        
        if len(actual_items) == 0:
            continue
        
        # Popularity-based recommendations
        recommended_items = set(
            recommend_popular_movies(
                user_id,
                n=k
            )
        )
        
        if len(recommended_items) == 0:
            continue
        
        # Calculate hits
        hits = (
            recommended_items
            & actual_items
        )
        
        # Precision
        precision = (
            len(hits)
            / len(recommended_items)
        )
        
        # Recall
        recall = (
            len(hits)
            / len(actual_items)
        )
        
        # F1
        if precision + recall > 0:
            f1 = (
                2 * precision * recall
                / (precision + recall)
            )
        else:
            f1 = 0
        
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        
        evaluated_users += 1
    
    return {
        "Users Evaluated": evaluated_users,
        "Precision@10": np.mean(precisions),
        "Recall@10": np.mean(recalls),
        "F1@10": np.mean(f1_scores)
    }

In [ ]:
popularity_results = evaluate_popularity_baseline(
    evaluation_users_final,
    k=10
)

popularity_results

In [ ]:
comparison_table = pd.DataFrame([
    {
        "Method": "Popularity Baseline",
        "Users Evaluated": popularity_results["Users Evaluated"],
        "Precision@10": popularity_results["Precision@10"],
        "Recall@10": popularity_results["Recall@10"],
        "F1@10": popularity_results["F1@10"]
    },
    {
        "Method": "Item-Based Collaborative Filtering",
        "Users Evaluated": evaluation_results_final["Users Evaluated"],
        "Precision@10": evaluation_results_final["Precision@10"],
        "Recall@10": evaluation_results_final["Recall@10"],
        "F1@10": evaluation_results_final["F1@10"]
    }
])

comparison_table

In [ ]:
import matplotlib.pyplot as plt

# Prepare data
methods = comparison_table["Method"]

precision_values = comparison_table["Precision@10"]
recall_values = comparison_table["Recall@10"]
f1_values = comparison_table["F1@10"]

# Precision@10
plt.figure(figsize=(8, 5))
plt.bar(methods, precision_values)
plt.ylabel("Precision@10")
plt.title("Precision@10 Comparison")
plt.ylim(0, max(precision_values) * 1.2)
plt.xticks(rotation=15)
plt.show()

In [ ]:
metrics = ["Precision@10", "Recall@10", "F1@10"]

x = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(9, 5))

plt.bar(
    x - width / 2,
    [
        popularity_results["Precision@10"],
        popularity_results["Recall@10"],
        popularity_results["F1@10"]
    ],
    width,
    label="Popularity Baseline"
)

plt.bar(
    x + width / 2,
    [
        evaluation_results_final["Precision@10"],
        evaluation_results_final["Recall@10"],
        evaluation_results_final["F1@10"]
    ],
    width,
    label="Item-Based CF"
)

plt.ylabel("Score")
plt.title("Recommendation Performance Comparison")
plt.xticks(x, metrics)
plt.ylim(0, 0.35)
plt.legend()
plt.show()

In [ ]:
# Save final evaluation results

comparison_table.to_csv(
    "data/recommender_evaluation_results.csv",
    index=False
)

print("Evaluation results saved successfully.")

In [ ]:
# Save performance comparison chart

metrics = ["Precision@10", "Recall@10", "F1@10"]

x = np.arange(len(metrics))
width = 0.35

plt.figure(figsize=(9, 5))

plt.bar(
    x - width / 2,
    [
        popularity_results["Precision@10"],
        popularity_results["Recall@10"],
        popularity_results["F1@10"]
    ],
    width,
    label="Popularity Baseline"
)

plt.bar(
    x + width / 2,
    [
        evaluation_results_final["Precision@10"],
        evaluation_results_final["Recall@10"],
        evaluation_results_final["F1@10"]
    ],
    width,
    label="Item-Based CF"
)

plt.ylabel("Score")
plt.title("Recommendation Performance Comparison")
plt.xticks(x, metrics)
plt.ylim(0, 0.35)
plt.legend()

plt.savefig(
    "data/recommender_performance_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
links = pd.read_csv("data/links.csv")

print(links.head())
print(links.columns)
print("Number of links:", len(links))

In [ ]:
movie_metadata = movies.merge(
    links,
    on="movieId",
    how="left"
)

movie_metadata.head()

In [ ]:
# Save merged movie metadata for Streamlit

metadata_file = "data/movie_metadata.csv"

movie_metadata.to_csv(
    metadata_file,
    index=False
)

print("Movie metadata saved successfully.")
print("File:", metadata_file)
print("Number of movies:", len(movie_metadata))

In [ ]:
# Test recommendation results with movie metadata

test_recommendations = recommendations_user1.copy()

# Remove title from recommendation results
# because movie_metadata already contains the official movie title
test_recommendations = test_recommendations.drop(
    columns=["title"],
    errors="ignore"
)

# Add movie metadata
test_recommendations = test_recommendations.merge(
    movie_metadata[
        ["movieId", "title", "genres", "imdbId", "tmdbId"]
    ],
    on="movieId",
    how="left"
)

# Display the result
test_recommendations[
    ["movieId", "title", "genres", "tmdbId", "score"]
]

In [ ]:
# ==========================================
# Save Item-Based Collaborative Filtering
# artifacts for Streamlit
# ==========================================

import pickle
import os

# Create models folder
os.makedirs("models", exist_ok=True)

# 1. Save item similarity information
np.save(
    "models/item_distances.npy",
    all_item_distances
)

np.save(
    "models/item_indices.npy",
    all_item_indices
)

# 2. Save movie ID -> matrix index mapping
with open("models/movie_to_index.pkl", "wb") as f:
    pickle.dump(movie_to_index, f)

# 3. Save index -> movie ID mapping
with open("models/index_to_movie.pkl", "wb") as f:
    pickle.dump(index_to_movie, f)

# 4. Save user history
with open("models/user_history_indices.pkl", "wb") as f:
    pickle.dump(user_history_indices, f)

print("================================")
print("MODEL ARTIFACTS SAVED")
print("================================")

print("Item distances:", all_item_distances.shape)
print("Item indices:", all_item_indices.shape)
print("Movie mappings:", len(movie_to_index))
print("User histories:", len(user_history_indices))

In [ ]:
# ==========================================
# Save movie metadata used by the CF model
# ==========================================

model_movie_ids = set(movie_to_index.keys())

streamlit_movie_metadata = movie_metadata[
    movie_metadata["movieId"].isin(model_movie_ids)
].copy()

streamlit_movie_metadata.to_csv(
    "models/movie_metadata.csv",
    index=False
)

print("================================")
print("STREAMLIT METADATA SAVED")
print("================================")

print("Movies saved:", len(streamlit_movie_metadata))
print()
print(streamlit_movie_metadata.head())

In [ ]:
# ==========================================
# Check saved model files
# ==========================================

import os

model_files = [
    "models/item_distances.npy",
    "models/item_indices.npy",
    "models/movie_to_index.pkl",
    "models/index_to_movie.pkl",
    "models/user_history_indices.pkl",
    "models/movie_metadata.csv"
]

print("Model files:")
print()

for file in model_files:
    if os.path.exists(file):
        size_mb = os.path.getsize(file) / (1024 * 1024)
        print(f"✓ {file:<40} {size_mb:.2f} MB")
    else:
        print(f"✗ {file} NOT FOUND")

In [ ]:
# ==========================================
# Test loading saved model artifacts
# ==========================================

import pickle
import numpy as np
import pandas as pd

# Load model artifacts
loaded_distances = np.load(
    "models/item_distances.npy"
)

loaded_indices = np.load(
    "models/item_indices.npy"
)

with open("models/movie_to_index.pkl", "rb") as f:
    loaded_movie_to_index = pickle.load(f)

with open("models/index_to_movie.pkl", "rb") as f:
    loaded_index_to_movie = pickle.load(f)

with open("models/user_history_indices.pkl", "rb") as f:
    loaded_user_history_indices = pickle.load(f)

loaded_movie_metadata = pd.read_csv(
    "models/movie_metadata.csv"
)

print("================================")
print("MODEL LOADING TEST")
print("================================")

print("Distances:", loaded_distances.shape)
print("Indices:", loaded_indices.shape)
print("Movie mappings:", len(loaded_movie_to_index))
print("User histories:", len(loaded_user_history_indices))
print("Movie metadata:", loaded_movie_metadata.shape)